In [1]:
#============================================
# Celda 1 — Imports y carga de parquets RAW
#============================================
import pandas as pd
import numpy as np
import os

AEMET_RAW   = '../../output/ambiental/01-raw/raw_aemet_estaciones.parquet'
WAQI_RAW    = '../../output/ambiental/01-raw/raw_waqi_ciudades.parquet'
OUTPUT_DIR  = '../../output/ambiental/02-silver/'

df_aemet = pd.read_parquet(AEMET_RAW)
df_waqi  = pd.read_parquet(WAQI_RAW)

print(f'AEMET RAW  → {df_aemet.shape[0]} filas | {df_aemet.shape[1]} columnas')
print(f'WAQI  RAW  → {df_waqi.shape[0]}  filas | {df_waqi.shape[1]} columnas')


AEMET RAW  → 10137 filas | 22 columnas
WAQI  RAW  → 30  filas | 18 columnas


In [2]:
#============================================
# Celda 2 — Limpieza WAQI: filtrar por frescura
#============================================
FECHA_CAPTURA = pd.Timestamp('today').normalize()
UMBRAL_DIAS   = 7  # máximo retraso aceptable

df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'])
df_waqi['dias_retraso'] = (FECHA_CAPTURA - df_waqi['timestamp'].dt.normalize()).dt.days

antes = len(df_waqi)
df_waqi_ok  = df_waqi[df_waqi['dias_retraso'] <= UMBRAL_DIAS].copy()
df_waqi_old = df_waqi[df_waqi['dias_retraso'] >  UMBRAL_DIAS].copy()

print(f'Ciudades totales:       {antes}')
print(f'✅ Dentro del umbral:   {len(df_waqi_ok)}')
print(f'⚠️  Fuera del umbral:   {len(df_waqi_old)}  (>= {UMBRAL_DIAS} días de retraso)')
print()
print('Ciudades descartadas por dato antiguo:')
print(df_waqi_old[['ciudad','nombre','timestamp','dias_retraso']].to_string(index=False))


Ciudades totales:       30
✅ Dentro del umbral:   0
⚠️  Fuera del umbral:   30  (>= 7 días de retraso)

Ciudades descartadas por dato antiguo:
       ciudad                                           nombre           timestamp  dias_retraso
       madrid                                           Madrid 2026-02-09 14:00:00            64
    barcelona           Barcelona (Eixample), Catalunya, Spain 2026-03-26 10:00:00            19
     valencia        Pista de Silla, València, Valencia, Spain 2026-02-13 02:00:00            60
      sevilla                          Ranilla, Sevilla, Spain 2026-03-25 18:00:00            20
       bilbao             Mazarredo, Bilbao, País Vasco, Spain 2026-03-26 09:00:00            19
     zaragoza                     El Picarral, Zaragoza, Spain 2026-03-26 09:00:00            19
       malaga                         Carranque, Malaga, Spain 2026-03-25 19:00:00            20
       murcia         San Basilio Murcia Ciudad, Murcia, Spain 2026-03-26 11:00:0

In [3]:
#============================================
# Celda 3 — Limpieza WAQI: filtrar ciudades no españolas
#============================================
# Bounding box España peninsular + islas (lat 27-44, lon -19 a 5)
BBOX = {'lat_min': 27.0, 'lat_max': 44.5, 'lon_min': -19.0, 'lon_max': 5.0}

df_waqi_ok['en_espana'] = (
    df_waqi_ok['lat'].between(BBOX['lat_min'], BBOX['lat_max']) &
    df_waqi_ok['lon'].between(BBOX['lon_min'], BBOX['lon_max'])
)

fuera = df_waqi_ok[~df_waqi_ok['en_espana']]
if len(fuera) > 0:
    print(f'⚠️  Ciudades fuera de España eliminadas: {len(fuera)}')
    print(fuera[['ciudad', 'nombre', 'lat', 'lon']].to_string(index=False))
else:
    print('✅ Todas las ciudades están dentro del bounding box de España')

df_waqi_clean = df_waqi_ok[df_waqi_ok['en_espana']].drop(columns=['en_espana', 'dias_retraso'])
print(f'\nWAQI limpio → {len(df_waqi_clean)} ciudades')


✅ Todas las ciudades están dentro del bounding box de España

WAQI limpio → 0 ciudades


In [4]:
#============================================
# Celda 4 — Limpieza WAQI: tipos, nulos y normalización
#============================================
COLS_NUMERICAS_WAQI = ['aqi', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO',
                        'temperatura', 'humedad', 'presion', 'viento']

# Forzar numérico (por si llegan como string)
for col in COLS_NUMERICAS_WAQI:
    df_waqi_clean[col] = pd.to_numeric(df_waqi_clean[col], errors='coerce')

# Valores físicamente imposibles → NaN
limites = {
    'aqi':        (0, 500),
    'NO2':        (0, 2000),
    'PM25':       (0, 1000),
    'PM10':       (0, 1000),
    'O3':         (0, 500),
    'SO2':        (0, 2000),
    'CO':         (0, 50),
    'temperatura':(-30, 55),
    'humedad':    (0, 100),
    'presion':    (870, 1085),
    'viento':     (0, 250),
}
for col, (vmin, vmax) in limites.items():
    mask = ~df_waqi_clean[col].between(vmin, vmax, inclusive='both')
    n = mask.sum()
    if n > 0:
        print(f'⚠️  {col}: {n} valores fuera de rango [{vmin}, {vmax}] → NaN')
    df_waqi_clean.loc[mask, col] = np.nan

# Añadir nivel AQI categórico (escala EPA)
def nivel_aqi(v):
    if pd.isna(v):   return np.nan
    if v <= 50:      return 'Bueno'
    if v <= 100:     return 'Moderado'
    if v <= 150:     return 'No saludable grupos sensibles'
    if v <= 200:     return 'No saludable'
    if v <= 300:     return 'Muy no saludable'
    return 'Peligroso'

df_waqi_clean['nivel_aqi'] = df_waqi_clean['aqi'].apply(nivel_aqi)

print(f'\nWAQI limpio — Shape: {df_waqi_clean.shape}')
print('\nNulos por columna:')
print(df_waqi_clean[COLS_NUMERICAS_WAQI].isnull().sum())



WAQI limpio — Shape: (0, 18)

Nulos por columna:
aqi            0
NO2            0
PM25           0
PM10           0
O3             0
SO2            0
CO             0
temperatura    0
humedad        0
presion        0
viento         0
dtype: int64


In [5]:
#============================================
# Celda 5 — Asignar provincia a estaciones AEMET
#============================================
# El catálogo oficial AEMET (idema → provincia) se puede descargar en:
# https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones
# Mientras no esté disponible, usamos asignación por prefijo de idema
# Los primeros 2 dígitos del idema corresponden al código INE de provincia

# Diccionario código_prefijo → nombre_provincia → código_ine_provincia
PROV_MAP = {
    '00': ('Tarragona',    '43'), '01': ('Álava',         '01'),
    '02': ('Albacete',     '02'), '03': ('Alicante',      '03'),
    '04': ('Almería',      '04'), '05': ('Ávila',         '05'),
    '06': ('Badajoz',      '06'), '07': ('Baleares',      '07'),
    '08': ('Barcelona',    '08'), '09': ('Burgos',        '09'),
    '10': ('Cáceres',      '10'), '11': ('Cádiz',         '11'),
    '12': ('Castellón',    '12'), '13': ('Ciudad Real',   '13'),
    '14': ('Córdoba',      '14'), '15': ('A Coruña',      '15'),
    '16': ('Cuenca',       '16'), '17': ('Girona',        '17'),
    '18': ('Granada',      '18'), '19': ('Guadalajara',   '19'),
    '20': ('Gipuzkoa',     '20'), '21': ('Huelva',        '21'),
    '22': ('Huesca',       '22'), '23': ('Jaén',          '23'),
    '24': ('León',         '24'), '25': ('Lleida',        '25'),
    '26': ('La Rioja',     '26'), '27': ('Lugo',          '27'),
    '28': ('Madrid',       '28'), '29': ('Málaga',        '29'),
    '30': ('Murcia',       '30'), '31': ('Navarra',       '31'),
    '32': ('Ourense',      '32'), '33': ('Asturias',      '33'),
    '34': ('Palencia',     '34'), '35': ('Las Palmas',    '35'),
    '36': ('Pontevedra',   '36'), '37': ('Salamanca',     '37'),
    '38': ('S.C. Tenerife','38'), '39': ('Cantabria',     '39'),
    '40': ('Segovia',      '40'), '41': ('Sevilla',       '41'),
    '42': ('Soria',        '42'), '43': ('Tarragona',     '43'),
    '44': ('Teruel',       '44'), '45': ('Toledo',        '45'),
    '46': ('Valencia',     '46'), '47': ('Valladolid',    '47'),
    '48': ('Bizkaia',      '48'), '49': ('Zamora',        '49'),
    '50': ('Zaragoza',     '50'), '51': ('Ceuta',         '51'),
    '52': ('Melilla',      '52'), '60': ('Ceuta',         '51'),
    '70': ('Islas Baleares','07'),
    '90': ('Andalucía',    '18'),  # estaciones sin prefijo claro → Granada por defecto
    '99': ('Sin asignar',  'XX'),
    'C':  ('A Coruña',     '15'),  # prefijos letra
    'B':  ('Barcelona',    '08'),
    'Y':  ('Sin asignar',  'XX'),
}

def asignar_provincia(idema):
    prefijo = str(idema)[:2]
    if prefijo in PROV_MAP:
        return PROV_MAP[prefijo]
    prefijo1 = str(idema)[:1]
    if prefijo1 in PROV_MAP:
        return PROV_MAP[prefijo1]
    return ('Sin asignar', 'XX')

df_aemet[['provincia', 'cod_provincia']] = df_aemet['idema'].apply(
    lambda x: pd.Series(asignar_provincia(x))
)

total     = len(df_aemet)
asignadas = (df_aemet['cod_provincia'] != 'XX').sum()
sin_prov  = (df_aemet['cod_provincia'] == 'XX').sum()

print(f'Total estaciones:       {total}')
print(f'✅ Con provincia asignada: {asignadas} ({asignadas/total*100:.1f}%)')
print(f'⚠️  Sin provincia:          {sin_prov}  ({sin_prov/total*100:.1f}%)')
print()
print('Distribución por provincia (top 10 en nº estaciones):')
print(df_aemet.groupby('provincia').size().sort_values(ascending=False).head(10))


Total estaciones:       10137
✅ Con provincia asignada: 6672 (65.8%)
⚠️  Sin provincia:          3465  (34.2%)

Distribución por provincia (top 10 en nº estaciones):
provincia
Sin asignar    3465
A Coruña        982
Barcelona       533
Cáceres         398
Ceuta           277
Cádiz           272
Tarragona       224
Córdoba         222
Navarra         212
Ciudad Real     190
dtype: int64


In [6]:
#============================================
# Celda 6 — Limpieza AEMET: separar core vs auxiliares
#============================================
COLS_CORE = ['idema', 'ubi', 'lat', 'lon', 'alt', 'fecha',
             'provincia', 'cod_provincia',
             'ta', 'tamin', 'tamax', 'prec', 'hr']
COLS_AUX  = ['vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar',
             'vis', 'inso', 'ts', 'tpr']

df_aemet_core = df_aemet[COLS_CORE].copy()
df_aemet_aux  = df_aemet[['idema', 'fecha'] + COLS_AUX].copy()

# Forzar tipos numéricos en core
for col in ['ta', 'tamin', 'tamax', 'prec', 'hr']:
    df_aemet_core[col] = pd.to_numeric(df_aemet_core[col], errors='coerce')

# Validar rangos físicos core
limites_aemet = {
    'ta':    (-40, 55),
    'tamin': (-40, 55),
    'tamax': (-40, 55),
    'prec':  (0, 300),   # mm/día — récord España ~280mm
    'hr':    (0, 100),
}
for col, (vmin, vmax) in limites_aemet.items():
    mask = ~df_aemet_core[col].between(vmin, vmax, inclusive='both')
    n = mask.sum()
    if n > 0:
        print(f'⚠️  {col}: {n} valores fuera de rango [{vmin}, {vmax}] → NaN')
    df_aemet_core.loc[mask, col] = np.nan

print(f'AEMET core  → {df_aemet_core.shape}')
print(f'AEMET aux   → {df_aemet_aux.shape}')
print()
print('Nulos AEMET core por columna:')
nulos_core = df_aemet_core[['ta','tamin','tamax','prec','hr']].isnull().sum()
nulos_pct  = (nulos_core / len(df_aemet_core) * 100).round(2)
print(pd.DataFrame({'nulos': nulos_core, '%': nulos_pct}))


⚠️  ta: 150 valores fuera de rango [-40, 55] → NaN
⚠️  tamin: 150 valores fuera de rango [-40, 55] → NaN
⚠️  tamax: 150 valores fuera de rango [-40, 55] → NaN
⚠️  prec: 272 valores fuera de rango [0, 300] → NaN
⚠️  hr: 152 valores fuera de rango [0, 100] → NaN
AEMET core  → (10137, 13)
AEMET aux   → (10137, 12)

Nulos AEMET core por columna:
       nulos     %
ta       150  1.48
tamin    150  1.48
tamax    150  1.48
prec     272  2.68
hr       152  1.50


In [7]:
#============================================
# Celda 7 — Agregar AEMET a nivel provincia × fecha
#============================================
df_aemet_core['fecha'] = pd.to_datetime(df_aemet_core['fecha'])

# Excluir estaciones sin provincia asignada
df_prov = df_aemet_core[df_aemet_core['cod_provincia'] != 'XX'].copy()

df_aemet_prov = df_prov.groupby(['cod_provincia', 'provincia', 'fecha']).agg(
    n_estaciones   = ('idema',  'count'),
    ta_media       = ('ta',     'mean'),
    tamin_media    = ('tamin',  'mean'),
    tamax_media    = ('tamax',  'mean'),
    prec_total     = ('prec',   'sum'),    # precipitación: suma provincial
    hr_media       = ('hr',     'mean'),
    ta_std         = ('ta',     'std'),    # dispersión intra-provincial
).reset_index()

# Redondear
for col in ['ta_media','tamin_media','tamax_media','prec_total','hr_media','ta_std']:
    df_aemet_prov[col] = df_aemet_prov[col].round(2)

print(f'AEMET agregado provincia×fecha → {df_aemet_prov.shape}')
print()
print(df_aemet_prov.head(10).to_string(index=False))


AEMET agregado provincia×fecha → (41, 10)

cod_provincia      provincia      fecha  n_estaciones  ta_media  tamin_media  tamax_media  prec_total  hr_media  ta_std
           01          Álava 2026-04-14            88      9.39         8.34         9.67         0.0     69.31    4.63
           02       Albacete 2026-04-14            75     12.41        11.68        12.64         0.0     72.04    4.64
           03       Alicante 2026-04-14           115      9.39         8.25         9.61         0.0     65.79    6.58
           04        Almería 2026-04-14            60     14.05        13.55        14.16         0.0     47.34    2.77
           07 Islas Baleares 2026-04-14           161     14.12        13.28        14.48         0.0     48.43    5.43
           08      Barcelona 2026-04-14           533     12.97        12.21        13.30         4.0     75.03    3.66
           10        Cáceres 2026-04-14           398     10.19         9.47        10.34         9.5     86.42    3.

In [8]:
#============================================
# Celda 8 — Agregar AEMET a nivel provincia × año
#============================================
df_aemet_prov['anio'] = df_aemet_prov['fecha'].dt.year

df_aemet_anual = df_aemet_prov.groupby(['cod_provincia', 'provincia', 'anio']).agg(
    ta_media_anual    = ('ta_media',    'mean'),
    tamin_media_anual = ('tamin_media', 'mean'),
    tamax_media_anual = ('tamax_media', 'mean'),
    prec_total_anual  = ('prec_total',  'sum'),
    hr_media_anual    = ('hr_media',    'mean'),
    dias_con_datos    = ('n_estaciones','count'),
).reset_index()

for col in df_aemet_anual.select_dtypes('float').columns:
    df_aemet_anual[col] = df_aemet_anual[col].round(2)

print(f'AEMET anual provincia×año → {df_aemet_anual.shape}')
print()
print(df_aemet_anual.head(10).to_string(index=False))


AEMET anual provincia×año → (41, 9)

cod_provincia      provincia  anio  ta_media_anual  tamin_media_anual  tamax_media_anual  prec_total_anual  hr_media_anual  dias_con_datos
           01          Álava  2026            9.39               8.34               9.67               0.0           69.31               1
           02       Albacete  2026           12.41              11.68              12.64               0.0           72.04               1
           03       Alicante  2026            9.39               8.25               9.61               0.0           65.79               1
           04        Almería  2026           14.05              13.55              14.16               0.0           47.34               1
           07 Islas Baleares  2026           14.12              13.28              14.48               0.0           48.43               1
           08      Barcelona  2026           12.97              12.21              13.30               4.0           75.03       

In [9]:
#============================================
# Celda 9 — Resumen calidad de datos limpios
#============================================
def resumen_calidad(nombre, df, cols_num):
    total_celdas = df[cols_num].size
    nulos_total  = df[cols_num].isnull().sum().sum()
    pct_nulos    = round(nulos_total / total_celdas * 100, 1)
    print(f'\n📋 {nombre}')
    print(f'   Filas: {len(df)} | Columnas: {len(df.columns)} | Nulos: {pct_nulos}%')
    nulos_col = df[cols_num].isnull().sum()
    nulos_col = nulos_col[nulos_col > 0]
    if len(nulos_col):
        print('   Nulos por columna:')
        for col, n in nulos_col.items():
            print(f'     {col}: {n} ({n/len(df)*100:.1f}%)')

resumen_calidad('AEMET core (estaciones)',
                df_aemet_core,
                ['ta','tamin','tamax','prec','hr'])

resumen_calidad('AEMET agregado (provincia×fecha)',
                df_aemet_prov,
                ['ta_media','tamin_media','tamax_media','prec_total','hr_media'])

resumen_calidad('AEMET anual (provincia×año)',
                df_aemet_anual,
                ['ta_media_anual','tamin_media_anual','tamax_media_anual',
                 'prec_total_anual','hr_media_anual'])

resumen_calidad('WAQI ciudades limpias',
                df_waqi_clean,
                ['aqi','NO2','PM25','PM10','O3','SO2','CO'])



📋 AEMET core (estaciones)
   Filas: 10137 | Columnas: 13 | Nulos: 1.7%
   Nulos por columna:
     ta: 150 (1.5%)
     tamin: 150 (1.5%)
     tamax: 150 (1.5%)
     prec: 272 (2.7%)
     hr: 152 (1.5%)

📋 AEMET agregado (provincia×fecha)
   Filas: 41 | Columnas: 11 | Nulos: 0.0%

📋 AEMET anual (provincia×año)
   Filas: 41 | Columnas: 9 | Nulos: 0.0%

📋 WAQI ciudades limpias
   Filas: 0 | Columnas: 18 | Nulos: nan%


/tmp/ipykernel_23988/1103464176.py:7: RuntimeWarning: invalid value encountered in scalar divide
  pct_nulos    = round(nulos_total / total_celdas * 100, 1)


In [10]:
#============================================
# Celda 10 — Guardar parquets limpios
#============================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

outputs = {
    'clean_aemet_estaciones_core.parquet':  df_aemet_core,
    'clean_aemet_estaciones_aux.parquet':   df_aemet_aux,
    'clean_aemet_provincia_fecha.parquet':  df_aemet_prov,
    'clean_aemet_provincia_anio.parquet':   df_aemet_anual,
    'clean_waqi_ciudades.parquet':          df_waqi_clean,
}

for filename, df in outputs.items():
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_parquet(path, index=False)
    print(f'✅ Guardado: {path} ({len(df)} filas)')

print('\n🎉 Todos los parquets ambiental limpios guardados en output/')


✅ Guardado: ../../output/ambiental/02-silver/clean_aemet_estaciones_core.parquet (10137 filas)
✅ Guardado: ../../output/ambiental/02-silver/clean_aemet_estaciones_aux.parquet (10137 filas)
✅ Guardado: ../../output/ambiental/02-silver/clean_aemet_provincia_fecha.parquet (41 filas)
✅ Guardado: ../../output/ambiental/02-silver/clean_aemet_provincia_anio.parquet (41 filas)
✅ Guardado: ../../output/ambiental/02-silver/clean_waqi_ciudades.parquet (0 filas)

🎉 Todos los parquets ambiental limpios guardados en output/
